In [1]:
!pip install langchain langchain-community langchain-text-splitters \
             langchain-core sentence-transformers faiss-cpu pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Replace 'BFL' with your actual folder name exactly as it appears in Drive
folder_path = '/content/drive/MyDrive/BFL'

# Confirm it exists and see what's inside
os.listdir(folder_path)

['AI & IP in Education_October 2025.pdf',
 'Alberta Colleges - 2025 Symposium - FM Presentation (final).pdf',
 'ACUTI Claims 101 Oct 2025.pdf',
 'AB Colleges 2025 Symposium - Edmonton October 2025 - Presentation.pdf']

In [4]:
# LOADER
from langchain_community.document_loaders import PyPDFDirectoryLoader

# CHUNKER — moved to langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# EMBEDDER + VECTOR STORE
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# GENERATOR
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/tmp/ipykernel_2091/2897637619.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [5]:
# Load all PDFs from your Drive folder
loader = PyPDFDirectoryLoader(folder_path)
docs = loader.load()
print(f"Loaded {len(docs)} pages")

# Chunk them
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

Loaded 146 pages
Created 237 chunks


In [6]:
# Embed and store in FAISS
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 12})

print("Vector store ready")

/tmp/ipykernel_2091/3143059132.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store ready


In [7]:
# Prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

# Format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [8]:
!pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.0 MB/s eta 0:00:00


In [9]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get('grokapi')
)

print("LLM ready")

LLM ready


In [10]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke("what is covered for water damage?with the source ")
print(answer)

According to the context, water damage is covered with a deductible of 5% minimum $50,000, subject to a maximum of $150,000. 

Source: Page 13 BFL CANADA Risk and Insurance Services Inc. | Financial Services Firm | bflcanada.ca 
 Property Insurance (continued) 
o Deductibles (highlighting the following certain deductibles and other deductibles not mentioned apply):
• $50,000 All Other Losses (AOP); except:
...
- 5% minimum $50,000 subject to maximum $150,000 Water Damage


# **Pydantic schema**

Cell 1 — define the schema:

In [11]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="Answer based only on the retrieved context")
    cited_chunk_ids: list[str] = Field(description="IDs of chunks used to answer")
    confidence: Literal["high", "medium", "low"] = Field(description="Confidence based on context quality")

Cell 2 — bind it to your LLM:

In [12]:
structured_llm = llm.with_structured_output(RAGResponse)

Cell 3 — rebuild the chain with structured output:

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

structured_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context.
Return your answer as a JSON object with:
- answer: your answer
- cited_chunk_ids: list of chunk identifiers you used (use format "chunk_0", "chunk_1" etc based on order in context)
- confidence: "high" if context directly answers, "medium" if partial, "low" if context is weak

Context:
{context}

Question: {question}
""")

structured_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | structured_prompt
    | structured_llm
)

response = structured_chain.invoke("what is covered for water damage?")
print(response)
print(type(response))

answer='Water damage is covered with a deductible of 5% minimum $50,000 subject to a maximum of $150,000' cited_chunk_ids=['chunk_6', 'chunk_13'] confidence='high'
<class '__main__.RAGResponse'>


In [14]:
def format_docs(docs):
    return "\n\n".join(
        f"[chunk_{i}] (source: {doc.metadata.get('source', 'unknown')}, page: {doc.metadata.get('page', '?')})\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

structured_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context.
Return your answer as a JSON object with:
- answer: your answer
- cited_chunk_ids: list of chunk identifiers you used (use format "chunk_0", "chunk_1" etc based on order in context)
- confidence: "high" if context directly answers, "medium" if partial, "low" if context is weak

Context:
{context}

Question: {question}
""")

structured_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | structured_prompt
    | structured_llm
)

response = structured_chain.invoke("what is covered for water damage?")
print(response)
print(type(response))

answer='Water damage is covered with a deductible of 5% minimum $50,000 subject to maximum $150,000' cited_chunk_ids=['chunk_9'] confidence='high'
<class '__main__.RAGResponse'>


# **LangSmith**

Cell 1 — enable tracing (just environment variables):

In [16]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LangSmithKey')
os.environ["LANGCHAIN_PROJECT"] = "rag-three-ways"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

print("Tracing enabled:", os.environ["LANGCHAIN_TRACING_V2"])
print("Project:", os.environ["LANGCHAIN_PROJECT"])

Tracing enabled: true
Project: rag-three-ways


Cell 2 — run your chain exactly as before:

In [17]:
response = structured_chain.invoke("what is covered for water damage?")
print(response)

answer='Water damage is covered with a deductible of 5% minimum $50,000 subject to maximum $150,000, under the Property Insurance policy.' cited_chunk_ids=['chunk_9'] confidence='high'


Another Test

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

test_chain = (
    ChatPromptTemplate.from_template("Say hello to {name}")
    | llm
    | StrOutputParser()
)

result = test_chain.invoke({"name": "Victoria"})
print(result)

Hello Victoria, it's nice to meet you. Is there something I can help you with or would you like to chat?


In [24]:
!pip install langsmith -q

Trace it in the Code

In [30]:
import time

start = time.time()
response = structured_chain.invoke("what is covered for water damage?")
total = time.time() - start

print(f"Total latency: {total:.2f}s")
print(response)

Total latency: 0.62s
answer='Water damage is covered with a deductible of $50,000, except for certain cases such as Keyano College Properties, Earthquake, and Musical instruments, which have different deductibles. Additionally, there is a 5% minimum $50,000 subject to maximum $150,000 deductible for Water Damage.' cited_chunk_ids=['chunk_9'] confidence='high'


In [31]:
# Retrieval only
start = time.time()
docs = retriever.invoke("what is covered for water damage?")
retrieval_time = time.time() - start

# Generation only
start = time.time()
response = structured_chain.invoke("what is covered for water damage?")
gen_time = (time.time() - start) - retrieval_time

print(f"Retrieval: {retrieval_time:.2f}s")
print(f"Generation: {gen_time:.2f}s")

Retrieval: 0.02s
Generation: 0.58s


Tracing showed retrieval was 0.02s and generation was 0.58s — so optimizing chunking or k wouldn't have moved p95. The lever is the model or caching.